# Chapter 1 — Your First Look at Indian Inflation

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch01_first_look_at_inflation.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 1.1 Why Python?

## 1.2 Three ways to run Python

## 1.3 Python as an economist's calculator

In [ ]:
nominal_growth = 0.098      # 9.8 per cent, written as a decimal
inflation = 0.034           # 3.4 per cent

real_growth = (1 + nominal_growth) / (1 + inflation) - 1
real_growth

In [ ]:
print(f"Real growth was {real_growth:.2%}")

In [ ]:
import math                       # brings in Python's maths functions

for g in [4, 6, 7, 8, 10]:        # repeat the next lines for each growth rate
    approx = 70 / g
    exact = math.log(2) / math.log(1 + g / 100)
    print(f"At {g:>2}% growth, GDP doubles in {approx:4.1f} years "
          f"(exact: {exact:4.1f})")

## 1.4 Your first real data: the Consumer Price Index

In [ ]:
import pandas as pd                       # the standard library for tables
from indiadata import mospi_cpi           # the book's data helper

cpi = mospi_cpi(sector="combined")        # All-India, General index
cpi.head()                                # show the first five rows

In [ ]:
print("Rows:", len(cpi))
print("From", cpi["date"].min().strftime("%B %Y"),
      "to", cpi["date"].max().strftime("%B %Y"))

## 1.5 Compute inflation yourself — then check it

In [ ]:
cpi["my_inflation"] = cpi["index"].pct_change(12) * 100
cpi[["date", "index", "inflation", "my_inflation"]].tail(6).round(
    {"inflation": 2, "my_inflation": 2})

In [ ]:
gap = (cpi["my_inflation"] - cpi["inflation"]).abs()
print(f"Largest gap between our inflation and MoSPI's: {gap.max():.2f} points")

In [ ]:
cpi[(cpi["date"] >= "2014-01-01") & cpi["inflation"].isna()]

## 1.6 Has the RBI met its target?

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.axhspan(2, 6, color="tab:green", alpha=0.12, label="Tolerance band (2–6%)")
ax.axhline(4, color="tab:green", linestyle="--", linewidth=1,
           label="Target (4%)")
ax.plot(cpi["date"], cpi["inflation"], color="tab:blue", linewidth=1.8,
        label="CPI-Combined inflation")

ax.set_title("India: consumer price inflation and the RBI's target",
             loc="left", fontsize=13)
ax.set_ylabel("Year-on-year, %")
ax.legend(loc="upper right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Source: MoSPI, CPI (base 2012=100), via eSankhyiki API.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
since_fit = cpi[cpi["date"] >= "2016-10-01"].dropna(subset=["inflation"])

inside = since_fit["inflation"].between(2, 6)
above = since_fit["inflation"] > 6
below = since_fit["inflation"] < 2

print(f"Months since Oct 2016 (excluding Apr–May 2020): {len(since_fit)}")
print(f"  inside the 2–6% band: {inside.mean():.0%}")
print(f"  above 6%:             {above.mean():.0%}")
print(f"  below 2%:             {below.mean():.0%}")
print(f"Average inflation:      {since_fit['inflation'].mean():.2f}%")

In [ ]:
quarterly = (since_fit.set_index("date")["inflation"]
             .resample("QE").mean())           # average of each quarter
outside = (quarterly > 6) | (quarterly < 2)

# count runs of consecutive quarters outside the band
run = outside.groupby((~outside).cumsum()).cumsum()
breaches = quarterly[run >= 3]
print("Quarters completing a run of 3+ quarters outside the band:")
print(breaches.round(2).to_string() if len(breaches) else "  none")

## 1.7 Bonus: 60 years of Indian growth in one chart

In [ ]:
from indiadata import worldbank

gdp = worldbank("NY.GDP.MKTP.KD.ZG")      # GDP growth, annual %, India
gdp.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ["tab:red" if v < 0 else "tab:blue" for v in gdp["value"]]
ax.bar(gdp["year"], gdp["value"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(1991, color="grey", linestyle=":", linewidth=1)
ax.text(1991.5, ax.get_ylim()[1] * 0.9, "1991 reforms", fontsize=9,
        color="grey")
ax.set_title("India: real GDP growth, annual %", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Source: World Bank, World Development Indicators "
         "(NY.GDP.MKTP.KD.ZG). Years are calendar-year based as reported by "
         "the World Bank.", fontsize=8, color="grey")
plt.show()

In [ ]:
before = gdp[gdp["year"].between(1961, 1990)]["value"].mean()
after = gdp[gdp["year"].between(1992, 2019)]["value"].mean()
worst = gdp.loc[gdp["value"].idxmin()]
print(f"Average growth 1961–1990: {before:.1f}%")
print(f"Average growth 1992–2019: {after:.1f}%")
print(f"Worst year: {int(worst['year'])} ({worst['value']:.1f}%)")

## Key ideas

## Exercises

## Further reading